# Data Processing Pipelines
Companion to the data processing notes. Run in order; later cells use the split from the first.

## Split first: load the loan data and hold out a test set

In [ ]:
import sys
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

ld = pd.read_csv("../../data/loan_data_train.csv")
ld = ld.drop(columns=["ID", "Amount.Funded.By.Investors"])        # leakage: decided after application
ld["Interest.Rate"] = pd.to_numeric(ld["Interest.Rate"].str.replace("%", ""), errors="coerce")
ld = ld.dropna(subset=["Interest.Rate"])                           # rows without a target are no use

x_train, x_test, y_train, y_test = train_test_split(
    ld.drop(columns="Interest.Rate"), ld["Interest.Rate"], test_size=0.2, random_state=2)
print(x_train.shape, x_test.shape)

## Pulling numbers out of text

In [ ]:
def pct_to_num(s):                      # "27.56%" -> 27.56
    return pd.to_numeric(s.str.replace("%", "", regex=False), errors="coerce")

def fico_mid(s):                        # "720-724" -> 722.0
    parts = s.str.split("-", expand=True)
    return (pd.to_numeric(parts[0], errors="coerce") + pd.to_numeric(parts[1], errors="coerce")) / 2

def emp_years(s):                       # "10+ years" -> 10, "< 1 year" -> 0, "5 years" -> 5
    s = s.str.replace("10+", "10", regex=False).str.replace("< 1", "0", regex=False)
    return pd.to_numeric(s.str.extract(r"(\d+)")[0], errors="coerce")

print(fico_mid(pd.Series(["720-724", "690-694"])).tolist(), emp_years(pd.Series(["10+ years", "< 1 year", "3 years"])).tolist())

## A pipeline step written from scratch

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class MedianImputer(BaseEstimator, TransformerMixin):

    def fit(self, x, y=None):
        self.medians_ = x.median()           # learned from whatever data fit() sees: the training data
        return self

    def transform(self, x):
        return x.fillna(self.medians_)       # applied unchanged to any data

imp = MedianImputer().fit(x_train[["Monthly.Income"]])
print(imp.medians_.to_dict())
print(imp.transform(x_test[["Monthly.Income"]]).isna().sum().to_dict())

## The loan data pipeline with ml_utils.DataPipe

In [ ]:
sys.path.append("../../utils")
from ml_utils import DataPipe

dp = DataPipe(
    cat_to_dummies=["Loan.Length", "Loan.Purpose", "Home.Ownership", "State"],
    cat_to_num=["Amount.Requested", "Open.CREDIT.Lines", "Revolving.CREDIT.Balance"],
    simple_num=["Monthly.Income", "Inquiries.in.the.Last.6.Months"],
    custom_func_dict={"Debt.To.Income.Ratio": pct_to_num,
                      "FICO.Range": fico_mid,
                      "Employment.Length": emp_years})
dp.fit(x_train)
X_train = dp.transform(x_train)
X_test = dp.transform(x_test)
print(X_train.shape, X_test.shape, int(X_train.isna().sum().sum()), int(X_test.isna().sum().sum()))
print([c for c in X_train.columns if c.startswith(("Loan.Length", "Home.Ownership"))])

## The same with scikit-learn's ColumnTransformer, inside cross-validation

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

def apply_custom(df):
    return pd.DataFrame({"dti": pct_to_num(df["Debt.To.Income.Ratio"]),
                         "fico": fico_mid(df["FICO.Range"]),
                         "emp": emp_years(df["Employment.Length"])})

to_number = FunctionTransformer(lambda df: df.apply(pd.to_numeric, errors="coerce"))
numeric = ["Amount.Requested", "Open.CREDIT.Lines", "Revolving.CREDIT.Balance",
           "Monthly.Income", "Inquiries.in.the.Last.6.Months"]
categorical = ["Loan.Length", "Loan.Purpose", "Home.Ownership", "State"]
custom = ["Debt.To.Income.Ratio", "FICO.Range", "Employment.Length"]

preprocess = ColumnTransformer([
    ("num", make_pipeline(to_number, SimpleImputer(strategy="median"), StandardScaler()), numeric),
    ("cat", make_pipeline(SimpleImputer(strategy="constant", fill_value="__missing__"),
                          OneHotEncoder(min_frequency=0.01, handle_unknown="infrequent_if_exist")), categorical),
    ("custom", make_pipeline(FunctionTransformer(apply_custom), SimpleImputer(strategy="median"),
                             StandardScaler()), custom),
])
model = Pipeline([("prep", preprocess), ("reg", LinearRegression())])

scores = cross_val_score(model, x_train, y_train, cv=5, scoring="neg_mean_absolute_error")
print("CV MAE per fold:", (-scores).round(3), " mean:", round(-scores.mean(), 3))
model.fit(x_train, y_train)
print("test MAE:", round(np.mean(np.abs(model.predict(x_test) - y_test)), 3))

## Four scalers and one outlier

In [ ]:
from sklearn.preprocessing import MinMaxScaler, RobustScaler

x = np.array([[10.0], [12.0], [14.0], [15.0], [100.0]])
table = pd.DataFrame({
    "raw": x.ravel(),
    "standard": StandardScaler().fit_transform(x).ravel(),
    "minmax": MinMaxScaler().fit_transform(x).ravel(),
    "robust": RobustScaler().fit_transform(x).ravel()})
print(table.round(3))

## Cyclic months: December is next to January

In [ ]:
months = np.arange(1, 13)
angle = 2 * np.pi * months / 12
pts = np.c_[np.cos(angle), np.sin(angle)]
print(round(np.linalg.norm(pts[11] - pts[0]), 4), round(np.linalg.norm(pts[0] - pts[1]), 4), round(np.linalg.norm(pts[0] - pts[6]), 4))

## Try it yourself
1. Difference test: customer ID, number of children, credit score, blood group, year of birth, hour of day.
2. Write a `FrequencyEncoder` step (fit stores shares, transform maps; unseen categories get 0).
3. Cyclic coordinates of 23:00 and 01:00: how far apart vs 11:00 and 13:00?
4. Add 1000 as a sixth value to `[10, 12, 14, 15, 100]`: which scaler changes least for the ordinary values?
5. Replace `LinearRegression` with `Ridge` in the scikit-learn pipeline.

## Choosing features: correlation vs mutual information

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import mutual_info_regression

rng = np.random.default_rng(0)
n = 2000
demo = pd.DataFrame({"x": rng.uniform(-3, 3, n), "z": rng.normal(size=n), "noise": rng.normal(size=n)})
target = demo["x"] ** 2 + 0.5 * demo["z"] + rng.normal(0, 0.5, n)       # x matters through a U shape
print("correlation with target:", demo.corrwith(target).round(3).to_dict())
print("mutual information     :", {c: round(float(v), 3) for c, v in zip(demo.columns, mutual_info_regression(demo, target, random_state=0))})

On the loan data:

In [ ]:
ld = pd.read_csv("../../data/loan_data_train.csv")
ld["Interest.Rate"] = pd.to_numeric(ld["Interest.Rate"].str.replace("%", ""), errors="coerce")
ld["FICO"] = ld["FICO.Range"].str.split("-", expand=True).astype(float).mean(axis=1)
ld["Loan.Length"] = pd.to_numeric(ld["Loan.Length"].str.replace(" months", ""), errors="coerce")
cols = ["FICO", "Loan.Length", "Amount.Requested", "Inquiries.in.the.Last.6.Months", "Debt.To.Income.Ratio",
        "Monthly.Income", "Open.CREDIT.Lines", "Revolving.CREDIT.Balance"]
ld["Debt.To.Income.Ratio"] = pd.to_numeric(ld["Debt.To.Income.Ratio"].astype(str).str.replace("%", ""), errors="coerce")
for col in cols:
    ld[col] = pd.to_numeric(ld[col], errors="coerce")
ld = ld[cols + ["Interest.Rate"]].dropna()
X, y = ld[cols], ld["Interest.Rate"]
ranking = pd.DataFrame({"|correlation|": X.corrwith(y).abs(),
                        "mutual information": mutual_info_regression(X, y, random_state=0)}, index=cols)
print(ranking.sort_values("mutual information", ascending=False).round(3))

Redundant pairs:

In [ ]:
corr = X.corr()
pairs = [(a, b, corr.loc[a, b]) for i, a in enumerate(cols) for b in cols[i + 1:] if abs(corr.loc[a, b]) > 0.3]
for a, b, r in sorted(pairs, key=lambda t: -abs(t[2])):
    print(f"{a:32s} {b:28s} r = {r:.2f}")